# Meal Memory: Recipe Retrieval

Prepare a recipe collection and compare keyword-based search with
semantic search using pretrained sentence embeddings.

Recipe suggestions are related dishes, not the exact recipe recovered
from a photograph.

AI attribution: Initial code and explanations were drafted with
assistance from OpenAI ChatGPT. Changes and verification will be
documented in ATTRIBUTION.md.

In [1]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("/home/users/al581/meal-memory")
RECIPE_PATH = PROJECT_ROOT / "data" / "raw" / "full_dataset_complete.csv"

assert RECIPE_PATH.stat().st_size == 2_294_981_083, (
    "Wrong or incomplete dataset file."
)

assert RECIPE_PATH.is_file(), f"File not found: {RECIPE_PATH}"

print(f"CSV size: {RECIPE_PATH.stat().st_size / 1024**3:.2f} GiB")

recipe_preview = pd.read_csv(RECIPE_PATH, nrows=5)

print("\nColumns:")
print(recipe_preview.columns.tolist())

print("\nFirst recipe:")
for column, value in recipe_preview.iloc[0].items():
    print(f"{column}: {str(value)[:600]}")

with open(
    PROJECT_ROOT / "data" / "splits" / "food20_config.json"
) as file:
    food_config = json.load(file)

SELECTED_CLASSES = food_config["classes"]
print("\nLoaded food categories:", len(SELECTED_CLASSES))

CSV size: 2.14 GiB

Columns:
['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']

First recipe:
Unnamed: 0: 0
title: No-Bake Nut Cookies
ingredients: ["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]
directions: ["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]
link: www.cookbooks.com/Recipe-Details.aspx?id=44874
source: Gathered
NER: ["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]

Loaded food categories: 20


In [2]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import re
from collections import Counter

# These identify candidate recipes by title.
# They are retrieval hints, not verified dish labels.
DISH_PATTERNS = {
    "apple_pie": r"\bapple\s+pie\b",
    "caesar_salad": r"\bcaesar\b",
    "cheesecake": r"\bcheesecake\b",
    "chocolate_cake": r"\bchocolate\b.*\bcake\b",
    "french_toast": r"\bfrench\s+toast\b",
    "fried_rice": r"\bfried\s+rice\b",
    "grilled_cheese_sandwich": r"\bgrilled\s+cheese\b",
    "hamburger": r"\b(?:hamburgers?|burgers?)\b",
    "lasagna": r"\blasagn[ae]\b",
    "macaroni_and_cheese": (
        r"\b(?:macaroni|mac)\s+(?:and|n)\s+cheese\b"
    ),
    "omelette": r"\bomele(?:t|tte)s?\b",
    "pancakes": r"\bpancakes?\b",
    "pizza": r"\bpizzas?\b",
    "ramen": r"\bramen\b",
    "risotto": r"\brisotto\b",
    "spaghetti_bolognese": r"\bbolognese\b",
    "spaghetti_carbonara": r"\bcarbonara\b",
    "sushi": r"\bsushi\b",
    "tacos": r"\btacos?\b",
    "waffles": r"\bwaffles?\b",
}

assert set(DISH_PATTERNS) == set(SELECTED_CLASSES)

coverage = Counter()
title_examples = {name: [] for name in SELECTED_CLASSES}
total_rows = 0

# A parsing error will stop the cell rather than silently skip rows.
for chunk in pd.read_csv(
    RECIPE_PATH,
    usecols=["title"],
    chunksize=50_000,
):
    total_rows += len(chunk)

    titles = (
        chunk["title"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )

    for class_name, pattern in DISH_PATTERNS.items():
        matches = titles.str.contains(
            pattern, flags=re.IGNORECASE, regex=True, na=False
        )
        coverage[class_name] += int(matches.sum())

        remaining = 3 - len(title_examples[class_name])
        if remaining > 0:
            title_examples[class_name].extend(
                chunk.loc[matches, "title"].head(remaining).tolist()
            )

coverage_df = pd.DataFrame([
    {
        "class_name": name,
        "candidate_recipes": coverage[name],
        "example_titles": title_examples[name],
    }
    for name in SELECTED_CLASSES
])

print(f"CSV parsed successfully: {total_rows:,} recipe rows")
display(coverage_df)

RESULTS_DIR = PROJECT_ROOT / "docs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

coverage_df[["class_name", "candidate_recipes"]].to_csv(
    RESULTS_DIR / "recipe_title_coverage_initial.csv",
    index=False,
)

CSV parsed successfully: 2,231,142 recipe rows


,class_name,candidate_recipes,example_titles
0,apple_pie,5359,"[Apple Pie, All-American Apple Pie, Sour Cream..."
1,caesar_salad,2573,"[Caesar Salad, Quick And Easy Caesar Salad, Ca..."
2,cheesecake,15260,"[Cherry Topped Cheesecake, Grandma'S Cheesecak..."
3,chocolate_cake,16859,"[Easy German Chocolate Cake, Chocolate Peanut ..."
4,french_toast,4391,"[Baked French Toast, French Toast And Sauce, V..."
5,fried_rice,3147,"[Fried Rice, Japanese Fried Rice, Mike'S Fried..."
6,grilled_cheese_sandwich,1516,"[Grilled Cheese Sandwich, Jenna'S Grilled Chee..."
7,hamburger,18535,"[Vegetable-Burger Soup, Spanish Hamburgers, Ch..."
8,lasagna,11752,"[Lasagne, Sour Cream Lasagna, Don Hill'S Lasagna]"
9,macaroni_and_cheese,5680,[Thomas Jefferson'S Macaroni And Cheese Puddin...


In [3]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import ast
import hashlib
import random
import unicodedata

SEED = 386
PER_CLASS = 200

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def clean_text(value):
    """Normalize whitespace while preserving readable recipe text."""
    return " ".join(unicodedata.normalize("NFKC", str(value)).split())


def parse_text_list(value):
    """Parse serialized lists without executing their contents."""
    if not isinstance(value, str):
        return []

    try:
        parsed = json.loads(value)
    except (ValueError, TypeError):
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError, TypeError, RecursionError):
            return []

    if not isinstance(parsed, list):
        return []

    if not all(isinstance(item, str) for item in parsed):
        return []

    return [clean_text(item) for item in parsed if item.strip()]


compiled_patterns = {
    name: re.compile(pattern, re.IGNORECASE)
    for name, pattern in DISH_PATTERNS.items()
}

combined_pattern = "|".join(
    f"(?:{pattern})" for pattern in DISH_PATTERNS.values()
)

# Reservoir sampling prevents the earliest CSV rows from dominating.
rng = random.Random(SEED)
reservoirs = {name: [] for name in SELECTED_CLASSES}
valid_counts = Counter()
audit = Counter()
seen_fingerprints = set()

required_columns = [
    "Unnamed: 0", "title", "ingredients",
    "directions", "link", "source",
]

for chunk_number, chunk in enumerate(
    pd.read_csv(
        RECIPE_PATH,
        usecols=required_columns,
        dtype=str,
        keep_default_na=False,
        chunksize=50_000,
    ),
    start=1,
):
    audit["rows_scanned"] += len(chunk)

    normalized_titles = (
        chunk["title"]
        .str.lower()
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )

    candidates = chunk.loc[
        normalized_titles.str.contains(
            combined_pattern, regex=True, na=False
        )
    ]

    for _, row in candidates.iterrows():
        audit["title_candidates"] += 1

        title = clean_text(row["title"])
        normalized_title = re.sub(
            r"[^a-z0-9]+", " ", title.lower()
        ).strip()

        ingredients = parse_text_list(row["ingredients"])
        directions = parse_text_list(row["directions"])

        if not title or len(ingredients) < 2 or not directions:
            audit["invalid_or_insufficient_content"] += 1
            continue

        # Conservative content deduplication, not near-duplicate detection.
        canonical = json.dumps(
            [
                normalized_title,
                sorted(item.casefold() for item in ingredients),
            ],
            ensure_ascii=False,
        )
        fingerprint = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        if fingerprint in seen_fingerprints:
            audit["duplicate_title_ingredients"] += 1
            continue

        seen_fingerprints.add(fingerprint)

        matched_classes = [
            name for name, pattern in compiled_patterns.items()
            if pattern.search(normalized_title)
        ]

        record = {
            "recipe_id": fingerprint,
            "source_record_id": row["Unnamed: 0"],
            "title": title,
            "ingredients": ingredients,
            "directions": directions,
            "source_link_raw": clean_text(row["link"]),
            "source": row["source"],
            "candidate_classes": matched_classes,
        }

        audit["unique_valid_candidates"] += 1

        for name in matched_classes:
            valid_counts[name] += 1
            bucket = reservoirs[name]

            if len(bucket) < PER_CLASS:
                bucket.append(record)
            else:
                replacement = rng.randrange(valid_counts[name])
                if replacement < PER_CLASS:
                    bucket[replacement] = record

    if chunk_number % 10 == 0:
        print(
            f"Scanned {audit['rows_scanned']:,} rows...",
            flush=True,
        )

# A recipe may match multiple categories; store it only once.
unique_records = {
    record["recipe_id"]: record
    for bucket in reservoirs.values()
    for record in bucket
}

recipe_records = sorted(
    unique_records.values(),
    key=lambda record: record["recipe_id"],
)

CANDIDATE_PATH = (
    PROCESSED_DIR / "recipe_candidates_v1.jsonl"
)

with CANDIDATE_PATH.open("w", encoding="utf-8") as file:
    for record in recipe_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

summary = pd.DataFrame([
    {
        "class_name": name,
        "unique_valid_title_matches": valid_counts[name],
        "sampled_candidates": len(reservoirs[name]),
    }
    for name in SELECTED_CLASSES
])

summary.to_csv(
    RESULTS_DIR / "recipe_candidate_sampling.csv",
    index=False,
)

with (RESULTS_DIR / "recipe_cleaning_audit.json").open("w") as file:
    json.dump(
        {
            "seed": SEED,
            "per_class_target": PER_CLASS,
            "source_file": RECIPE_PATH.name,
            "source_bytes": RECIPE_PATH.stat().st_size,
            "counts": dict(audit),
            "saved_unique_recipes": len(recipe_records),
        },
        file,
        indent=2,
    )

print("\nCleaning counts:", dict(audit))
print("Saved unique recipe candidates:", len(recipe_records))
print("Saved to:", CANDIDATE_PATH)
display(summary)

Scanned 500,000 rows...
Scanned 1,000,000 rows...
Scanned 1,500,000 rows...
Scanned 2,000,000 rows...

Cleaning counts: {'rows_scanned': 2231142, 'title_candidates': 151561, 'unique_valid_candidates': 151229, 'invalid_or_insufficient_content': 203, 'duplicate_title_ingredients': 129}
Saved unique recipe candidates: 3998
Saved to: /home/users/al581/meal-memory/data/processed/recipe_candidates_v1.jsonl


,class_name,unique_valid_title_matches,sampled_candidates
0,apple_pie,5350,200
1,caesar_salad,2571,200
2,cheesecake,15232,200
3,chocolate_cake,16788,200
4,french_toast,4382,200
5,fried_rice,3139,200
6,grilled_cheese_sandwich,1514,200
7,hamburger,18506,200
8,lasagna,11741,200
9,macaroni_and_cheese,5666,200


In [4]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import textwrap

# Load from the saved file so this also works after restarting.
CANDIDATE_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "recipe_candidates_v1.jsonl"
)

with CANDIDATE_PATH.open(encoding="utf-8") as file:
    recipe_records = [json.loads(line) for line in file if line.strip()]

recipes_df = pd.DataFrame(recipe_records)

for class_name in SELECTED_CLASSES:
    matching = recipes_df[
        recipes_df["candidate_classes"].apply(
            lambda names: class_name in names
        )
    ]

    sample = matching.sample(
        n=min(10, len(matching)),
        random_state=SEED,
    )

    print(f"\n{class_name.replace('_', ' ').upper()}")
    for title in sample["title"]:
        print(" •", textwrap.shorten(title, width=110, placeholder="…"))


APPLE PIE
 • Caramel Crunch Apple Pie
 • Grandma Ella'S Apple Pie
 • Sugar-Free Apple Pie
 • Apple Pie Cake
 • All-American Apple Pie
 • Diabetic Apple Pie
 • Swedish Apple Pie
 • Swedish Apple Pie
 • Loaf-Bread Apple Pie
 • Sugarless Apple Pie

CAESAR SALAD
 • Caesar Salad
 • Caesar Salad
 • Caesar Salad Dressing
 • Caesar Salad (For Two)
 • Quick And Easy Caesar Salad
 • Easy Garden Caesar Dip
 • Caesar Salad
 • Salmon Caesar
 • Caesar Salad With Sourdough Croutons
 • Caesar Salad

CHEESECAKE
 • German Chocolate Cheesecake
 • Easy Cheesecake
 • Very Berry Cheesecake
 • Creamy Cheesecake Recipe
 • Cheesecake
 • Philly Three-Step Cheesecake
 • Cheesecake
 • Cheesecake
 • German Chocolate Cheesecake
 • Kahlua Cheesecake

CHOCOLATE CAKE
 • Low Fat Chocolate Cake
 • Chocolate Sheet Cake
 • Chocolate Cake And Frosting
 • Chocolate Sheet Cake
 • Easy Cherry-Chocolate Cake
 • Chocolate Banana Cake
 • Chocolate Zucchini Cake
 • Chocolate Eclair Pudding Cake
 • Chocolate Cake
 • Kahlua Chocol

In [4]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
# Development rules based on the inspected title sample.
# These are heuristics, not verified recipe labels.

TITLE_EXCLUSIONS = {
    "apple_pie": (
        r"\b(?:paninis?|sandwich(?:es)?|smoothies?|"
        r"milkshakes?|filling|cocktails?)\b"
    ),
    "caesar_salad": (
        r"\b(?:dressings?|wraps?|sandwich(?:es)?|"
        r"stuffed mushrooms?|crazy bread|potato|pasta|"
        r"green beans?)\b"
    ),
    "cheesecake": (
        r"\b(?:cheesecake factory|bubble tea|"
        r"smoothies?|milkshakes?|dips?)\b"
    ),
    "grilled_cheese_sandwich": (
        r"\b(?:burgers?|soups?|dips?)\b"
    ),
    "hamburger": (
        r"\b(?:soups?|casseroles?|chow mein|"
        r"stroganoff|helpers?|dips?)\b"
    ),
    "pizza": (
        r"\b(?:fruit|dessert|cookie|cookies|"
        r"dip|dips|casseroles?|burgers?|"
        r"grilled cheese|soups?|salads?|"
        r"dough|sauce|seasoning)\b"
    ),
    "ramen": (
        r"\b(?:salads?|slaw|coleslaw|"
        r"stroganoff|to serve with)\b"
    ),
    "risotto": r"\bstuffed\b",
    "spaghetti_bolognese": (
        r"\b(?:lasagn[ae]|pizza|arancine|arancini)\b"
    ),
    "spaghetti_carbonara": (
        r"\b(?:ramen|mac and cheese|macaroni and cheese)\b"
    ),
    "sushi": (
        r"\b(?:dessert|s mores|smores|candy|"
        r"chocolate|salads?|vinegar)\b"
    ),
    "tacos": (
        r"\b(?:salads?|soups?|dips?|bowls?|"
        r"casseroles?|burgers?|rice|bake)\b"
    ),
    "waffles": r"\bsandwich(?:es)?\b",
}


def normalized_recipe_title(title):
    """Normalize title text for rule matching."""
    return re.sub(
        r"[^a-z0-9]+", " ", title.casefold()
    ).strip()


def keep_category_match(record, class_name):
    """Apply category-specific exclusions without deleting other tags."""
    title = normalized_recipe_title(record["title"])

    exclusion = TITLE_EXCLUSIONS.get(class_name)
    if exclusion and re.search(exclusion, title):
        return False

    # 'Caesar' alone can refer to unrelated dishes or brand names.
    if class_name == "caesar_salad":
        return bool(re.search(r"\bsalad\b", title))

    return True


filtered_records = []
filter_counts = []

for record in recipe_records:
    retained_classes = [
        name for name in record["candidate_classes"]
        if keep_category_match(record, name)
    ]

    if retained_classes:
        updated_record = dict(record)
        updated_record["candidate_classes"] = retained_classes
        filtered_records.append(updated_record)

for name in SELECTED_CLASSES:
    before = sum(
        name in record["candidate_classes"]
        for record in recipe_records
    )
    after = sum(
        name in record["candidate_classes"]
        for record in filtered_records
    )

    filter_counts.append({
        "class_name": name,
        "before_filter": before,
        "after_filter": after,
        "removed_matches": before - after,
    })

filter_summary = pd.DataFrame(filter_counts)

FILTERED_PATH = (
    PROCESSED_DIR / "recipe_candidates_filtered_v1.jsonl"
)

with FILTERED_PATH.open("w", encoding="utf-8") as file:
    for record in filtered_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

filter_summary.to_csv(
    RESULTS_DIR / "recipe_title_filter_summary.csv",
    index=False,
)

# Preserve the rules so preprocessing can be reproduced.
with (RESULTS_DIR / "recipe_title_filter_rules.json").open("w") as file:
    json.dump(
        {
            "title_exclusions": TITLE_EXCLUSIONS,
            "caesar_requires_word_salad": True,
            "status": "development_heuristics",
        },
        file,
        indent=2,
    )

print("Unique recipes before:", len(recipe_records))
print("Unique recipes after:", len(filtered_records))
display(filter_summary)

Unique recipes before: 3998
Unique recipes after: 3634


,class_name,before_filter,after_filter,removed_matches
0,apple_pie,203,186,17
1,caesar_salad,200,103,97
2,cheesecake,203,201,2
3,chocolate_cake,202,202,0
4,french_toast,204,204,0
5,fried_rice,200,200,0
6,grilled_cheese_sandwich,201,193,8
7,hamburger,216,191,25
8,lasagna,215,215,0
9,macaroni_and_cheese,203,203,0


In [6]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
# Reconstruct which recipes appeared in the earlier inspection.
previously_inspected_ids = set()

for name in SELECTED_CLASSES:
    matching = recipes_df[
        recipes_df["candidate_classes"].apply(
            lambda names: name in names
        )
    ]
    previous_sample = matching.sample(
        n=min(10, len(matching)),
        random_state=SEED,
    )
    previously_inspected_ids.update(previous_sample["recipe_id"])

filtered_df = pd.DataFrame(filtered_records)

for name in SELECTED_CLASSES:
    unseen = filtered_df[
        filtered_df["candidate_classes"].apply(
            lambda names: name in names
        )
        & ~filtered_df["recipe_id"].isin(previously_inspected_ids)
    ]

    sample = unseen.sample(
        n=min(5, len(unseen)),
        random_state=387,
    )

    print(f"\n{name.replace('_', ' ').upper()}")
    for title in sample["title"]:
        print(" •", title)


APPLE PIE
 • Norwegian Apple Pie
 • No Sugar Apple Pie(But Sweet As Pie)
 • Sour Cream Apple Pie
 • Delicious Sugar Apple Pie
 • No Fat-No Cholesterol-No Trouble Apple Pie

CAESAR SALAD
 • Caesar Salad
 • Caesar Salad(Serves 6)

CHEESECAKE
 • Easy Cheesecake
 • Cherry Topped Cheesecake
 • Cheesecake
 • The Best Cheesecake
 • No Bake Cheesecake

CHOCOLATE CAKE
 • Chocolate Caramel Cake
 • Dark Chocolate Cake
 • Easy Chocolate Cake
 • Easy Chocolate Cake
 • Mississippi Chocolate Chip Cake

FRENCH TOAST
 • Country French Toast
 • Creme Brule French Toast
 • French Toast
 • Puffy French Toast
 • Baked Apple French Toast

FRIED RICE
 • Mrs. Tran'S Fried Rice
 • Chicken Fried Rice
 • Japanese Fried Rice
 • Nicaraguan Fried Rice
 • Tofu Fried Rice

GRILLED CHEESE SANDWICH

HAMBURGER
 • Burger Beans
 • Kay'S Hamburger Cakes
 • Hamburger With Baked Beans
 • Hamburger And Macaroni
 • Nut Burgers

LASAGNA
 • Mexitalian Lasagna
 • Kathy'S Lasagna
 • Chicken Lasagna
 • Lasagna(World Famous!)
 • La

In [5]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
EXTRA_EXCLUSIONS = {
    "caesar_salad": r"\bburgers?\b",
    "cheesecake": r"\bice cream\b",
    "fried_rice": r"\bfried rice cakes?\b",
    "grilled_cheese_sandwich": r"\bcrackers?\b",
    "hamburger": (
        r"\b(?:buns?|quiche|one dish|one pot|"
        r"meatloaf|pies?)\b"
    ),
    "pancakes": r"\b(?:syrups?|mix|mixes)\b",
    "spaghetti_bolognese": r"\bdumplings?\b",
    "sushi": r"\b(?:krispie|krispies|marshmallows?)\b",
    "tacos": r"\b(?:pizzas?|sauce|bread|seasoning)\b",
    "waffles": r"\bsundaes?\b",
}

for name, extra_pattern in EXTRA_EXCLUSIONS.items():
    previous = TITLE_EXCLUSIONS.get(name)
    TITLE_EXCLUSIONS[name] = (
        f"(?:{previous})|(?:{extra_pattern})"
        if previous else extra_pattern
    )

# Apply the revised rules to the ORIGINAL candidates.
filtered_records_v2 = []

for record in recipe_records:
    retained = [
        name for name in record["candidate_classes"]
        if keep_category_match(record, name)
    ]

    if retained:
        updated = dict(record)
        updated["candidate_classes"] = retained
        filtered_records_v2.append(updated)

FILTERED_PATH_V2 = (
    PROCESSED_DIR / "recipe_candidates_filtered_v2.jsonl"
)

with FILTERED_PATH_V2.open("w", encoding="utf-8") as file:
    for record in filtered_records_v2:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

coverage_v2 = pd.DataFrame([
    {
        "class_name": name,
        "candidate_recipes": sum(
            name in record["candidate_classes"]
            for record in filtered_records_v2
        ),
    }
    for name in SELECTED_CLASSES
])

coverage_v2.to_csv(
    RESULTS_DIR / "recipe_filter_v2_coverage.csv",
    index=False,
)

with (RESULTS_DIR / "recipe_filter_v2_rules.json").open("w") as file:
    json.dump(
        {
            "title_exclusions": TITLE_EXCLUSIONS,
            "caesar_requires_word_salad": True,
            "status": "development_heuristics",
        },
        file,
        indent=2,
    )

print("Unique recipes after revision:", len(filtered_records_v2))
display(coverage_v2)

Unique recipes after revision: 3592


,class_name,candidate_recipes
0,apple_pie,186
1,caesar_salad,100
2,cheesecake,197
3,chocolate_cake,202
4,french_toast,204
5,fried_rice,198
6,grilled_cheese_sandwich,192
7,hamburger,174
8,lasagna,215
9,macaroni_and_cheese,203


In [8]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
inspection_classes = [
    "caesar_salad",
    "hamburger",
    "pizza",
    "ramen",
    "sushi",
    "tacos",
]

for name in inspection_classes:
    candidates = [
        record for record in filtered_records_v2
        if name in record["candidate_classes"]
    ]

    if not candidates:
        print(f"\nNo candidates for {name}")
        continue

    record = random.Random(388).choice(candidates)

    print(f"\n{'=' * 60}")
    print("Category:", name)
    print("Title:", record["title"])
    print("Ingredients:")
    for ingredient in record["ingredients"]:
        print(" -", ingredient)

    print("First two instruction steps:")
    for step in record["directions"][:2]:
        print(" -", step)


Category: caesar_salad
Title: Caesar Salad With Sourdough Croutons
Ingredients:
 - 1/2 c. fresh lemon juice
 - 2 Tbsp. Worcestershire sauce
 - 4 garlic cloves, coarsely chopped
 - 1/2 tsp. hot pepper sauce (such as Tabasco sauce)
 - 1 c. olive oil
 - 2 large heads romaine lettuce, torn into bite size pieces
 - 1 red onion, thinly sliced
 - Sourdough Croutons
 - 1 c. freshly grated Parmesan cheese (about 3 oz.)
First two instruction steps:
 - Blend first 4 ingredients in processor or blender. With machine running, gradually add oil in thin steady stream.
 - Season to taste with salt and pepper.

Category: hamburger
Title: Taco Burger
Ingredients:
 - 1 lb. ground meat
 - 1 pkg. taco seasoning
 - lettuce
 - tomatoes
 - rolls
 - cheese
First two instruction steps:
 - Brown meat.
 - Add taco seasoning according to directions. Spoon on roll.

Category: pizza
Title: Chocolate Pizza
Ingredients:
 - 1 (18 oz.) pkg. chocolate almond bark
 - 2 oz. (1 sq.) white almond bark
 - 2 c. miniature mars

In [6]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Reload the saved collection so this works after a restart.
with FILTERED_PATH_V2.open(encoding="utf-8") as file:
    search_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

search_df = pd.DataFrame(search_records).reset_index(drop=True)
assert len(search_df) == 3592, (
    "Wrong recipe collection loaded. Stop before building search."
)

assert search_df["recipe_id"].is_unique


def make_search_text(record):
    """Represent recipes consistently for both retrieval methods."""
    return (
        f"Recipe: {record['title']}. "
        f"Ingredients: {'; '.join(record['ingredients'])}"
    )


search_df["search_text"] = [
    make_search_text(record)
    for record in search_records
]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    search_df["search_text"]
)


def keyword_search(query, top_k=5):
    """Rank recipes by TF-IDF cosine similarity."""
    if not query.strip():
        raise ValueError("Enter a nonempty query.")

    query_vector = tfidf_vectorizer.transform([query])
    scores = linear_kernel(query_vector, tfidf_matrix).ravel()

    # Stable ordering makes ties reproducible.
    order = scores.argsort(kind="stable")[::-1]
    order = order[scores[order] > 0][:top_k]

    results = search_df.iloc[order][
        ["recipe_id", "title", "source_link_raw"]
    ].copy()
    results["score"] = scores[order]

    return results.reset_index(drop=True)


print("Searchable recipes:", len(search_df))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

display(keyword_search("ramen with pork and mushrooms"))
display(keyword_search("sushi with rice and vegetables"))
display(keyword_search("tacos with beef and cheese"))

Searchable recipes: 3592
TF-IDF matrix shape: (3592, 46473)


,recipe_id,title,source_link_raw,score
0,7a10d61f92fad66fbd2d03b2ecad497d928c274d30a5eb...,Ramen Pork ‘N’ Peppers,www.tasteofhome.com/recipes/ramen-pork-n-peppers/,0.244780
1,64f2801dfc6d64789e0ca4875138a78df2c557a44ad461...,Tokushima Ramen Pork Belly,cookpad.com/us/recipes/169302-tokushima-ramen-...,0.219332
2,43857ff9db20f6f3c7d469026216c9998afba567438120...,Bacon And Cabbage Ramen,www.food.com/recipe/bacon-and-cabbage-ramen-45...,0.127926
3,fce9cca46cc20f080e40f1511e29b59785a684260d4496...,Pork Ramen Alfredo-College Dorm Special,www.foodnetwork.com/recipes/robert-irvine/pork...,0.117199
4,39e80728e0916c291d623df8494a49a443d23479880b9c...,Black Pepper Pork With Ramen Noodles,www.food.com/recipe/black-pepper-pork-with-ram...,0.103687


,recipe_id,title,source_link_raw,score
0,9cf7a2b5252b12a4fc646a5d1bfdab339181a32bfee315...,Sushi Rice,www.food.com/recipe/sushi-rice-439085,0.290611
1,81e5ddfc6bfd3085938460c06fff2044c4c75eab7ed143...,Sushi Rice,www.food.com/recipe/sushi-rice-346823,0.224508
2,f8f27357a3b45d807014fc0333c76c8f0133ec9d192d0f...,Sushi Rice,www.food.com/recipe/sushi-rice-51594,0.205296
3,e4dbd357fc33492f78c5e82027d228a51064f1e13b033c...,Make-Ahead Sushi Rice,www.food.com/recipe/make-ahead-sushi-rice-190284,0.191593
4,7dea49088aba9c1258f23e3c3f36ef80b481e6d5350c0a...,Master Sushi Rice,www.foodnetwork.com/recipes/master-sushi-rice-...,0.178611


,recipe_id,title,source_link_raw,score
0,3b06b95864a71c54b60950ab9e63b4b4107244d1520a38...,Green Chile Beef Tacos,www.allrecipes.com/recipe/214894/green-chile-b...,0.178928
1,ba4a1b3996f7c48a1a267e8a7158d144f56857b1675c3d...,Navajo Tacos,www.cookbooks.com/Recipe-Details.aspx?id=172973,0.155328
2,4ba252a543d270c9bb191e4a16591c015fd8cd38703ca5...,Cream Tacos,www.cookbooks.com/Recipe-Details.aspx?id=651572,0.149126
3,1f61ff44b79a01494f95df3f05fb1675b97ac045438763...,Tico Tacos,www.cookbooks.com/Recipe-Details.aspx?id=204970,0.138909
4,b67034aec185b5655d303cd3e9a655dbd42e99707ef373...,Tina Tacos,www.cookbooks.com/Recipe-Details.aspx?id=229187,0.136760


In [10]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
try:
    import sentence_transformers
    print(
        "Sentence Transformers installed:",
        sentence_transformers.__version__,
    )
except ImportError:
    print("Sentence Transformers is not installed.")

/home/users/al581/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence Transformers installed: 6.3.0


In [11]:
#%pip install sentence-transformers

In [12]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import sentence_transformers
import torch
import torchvision

print("Sentence Transformers:", sentence_transformers.__version__)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

Sentence Transformers: 6.3.0
PyTorch: 2.15.0+cu132
Torchvision: 0.29.2+cu132
GPU available: True


In [8]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

recipe_embeddings = embedding_model.encode(
    search_df["search_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

assert recipe_embeddings.shape[0] == len(search_df)
assert np.isfinite(recipe_embeddings).all()

print("Embedding shape:", recipe_embeddings.shape)
print("Model input limit:", embedding_model.max_seq_length, "tokens")

Batches: 100%|██████████| 113/113 [00:02<00:00, 39.47it/s]


Embedding shape: (3592, 384)
Model input limit: 256 tokens


In [9]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
def semantic_search(query, top_k=5):
    """Rank recipes using normalized embedding similarity."""
    if not query.strip():
        raise ValueError("Enter a nonempty query.")

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    # Dot product equals cosine similarity for normalized embeddings.
    scores = recipe_embeddings @ query_embedding
    order = np.argsort(-scores, kind="stable")[:top_k]

    results = search_df.iloc[order][
        ["recipe_id", "title", "source_link_raw"]
    ].copy()
    results["score"] = scores[order]

    return results.reset_index(drop=True)


development_queries = [
    "ramen with pork and mushrooms",
    "sushi with rice and vegetables",
    "tacos with beef and cheese",
]

for query in development_queries:
    print(f"\nQUERY: {query}")

    print("KEYWORD SEARCH")
    display(keyword_search(query))

    print("SEMANTIC SEARCH")
    display(semantic_search(query))


QUERY: ramen with pork and mushrooms
KEYWORD SEARCH


,recipe_id,title,source_link_raw,score
0,7a10d61f92fad66fbd2d03b2ecad497d928c274d30a5eb...,Ramen Pork ‘N’ Peppers,www.tasteofhome.com/recipes/ramen-pork-n-peppers/,0.244780
1,64f2801dfc6d64789e0ca4875138a78df2c557a44ad461...,Tokushima Ramen Pork Belly,cookpad.com/us/recipes/169302-tokushima-ramen-...,0.219332
2,43857ff9db20f6f3c7d469026216c9998afba567438120...,Bacon And Cabbage Ramen,www.food.com/recipe/bacon-and-cabbage-ramen-45...,0.127926
3,fce9cca46cc20f080e40f1511e29b59785a684260d4496...,Pork Ramen Alfredo-College Dorm Special,www.foodnetwork.com/recipes/robert-irvine/pork...,0.117199
4,39e80728e0916c291d623df8494a49a443d23479880b9c...,Black Pepper Pork With Ramen Noodles,www.food.com/recipe/black-pepper-pork-with-ram...,0.103687


SEMANTIC SEARCH


,recipe_id,title,source_link_raw,score
0,6027797851de15fbc956c2215172d4db0321a99d0b9954...,Ramen Casserole,www.myrecipes.com/recipe/ramen-casserole,0.684124
1,196dbd2b5080c5d37d7f1b061e346f059e9a0253ce5252...,Turkey Ramen,www.epicurious.com/recipes/food/views/turkey-r...,0.648767
2,5a40825f80355d3f5e6c249916ccc7e3c849db8dcc5ee3...,One Pot Fancy Chicken Ramen,cookpad.com/us/recipes/340280-one-pot-fancy-ch...,0.640086
3,86f2ff3a7a0ea67e82f607ea415ab3740f9662a56a9777...,Ramen Party Mix,www.myrecipes.com/recipe/ramen-party-mix,0.630885
4,6deacfa4663a9cd6f4ba8122042d160c1ef5e216e58191...,Transform Sapporo Ichiban Ramen into Rich Tom ...,cookpad.com/us/recipes/157079-transform-sappor...,0.625382



QUERY: sushi with rice and vegetables
KEYWORD SEARCH


,recipe_id,title,source_link_raw,score
0,9cf7a2b5252b12a4fc646a5d1bfdab339181a32bfee315...,Sushi Rice,www.food.com/recipe/sushi-rice-439085,0.290611
1,81e5ddfc6bfd3085938460c06fff2044c4c75eab7ed143...,Sushi Rice,www.food.com/recipe/sushi-rice-346823,0.224508
2,f8f27357a3b45d807014fc0333c76c8f0133ec9d192d0f...,Sushi Rice,www.food.com/recipe/sushi-rice-51594,0.205296
3,e4dbd357fc33492f78c5e82027d228a51064f1e13b033c...,Make-Ahead Sushi Rice,www.food.com/recipe/make-ahead-sushi-rice-190284,0.191593
4,7dea49088aba9c1258f23e3c3f36ef80b481e6d5350c0a...,Master Sushi Rice,www.foodnetwork.com/recipes/master-sushi-rice-...,0.178611


... (2 outputs left out)


QUERY: tacos with beef and cheese
KEYWORD SEARCH


,recipe_id,title,source_link_raw,score
0,3b06b95864a71c54b60950ab9e63b4b4107244d1520a38...,Green Chile Beef Tacos,www.allrecipes.com/recipe/214894/green-chile-b...,0.178928
1,ba4a1b3996f7c48a1a267e8a7158d144f56857b1675c3d...,Navajo Tacos,www.cookbooks.com/Recipe-Details.aspx?id=172973,0.155328
2,4ba252a543d270c9bb191e4a16591c015fd8cd38703ca5...,Cream Tacos,www.cookbooks.com/Recipe-Details.aspx?id=651572,0.149126
3,1f61ff44b79a01494f95df3f05fb1675b97ac045438763...,Tico Tacos,www.cookbooks.com/Recipe-Details.aspx?id=204970,0.138909
4,b67034aec185b5655d303cd3e9a655dbd42e99707ef373...,Tina Tacos,www.cookbooks.com/Recipe-Details.aspx?id=229187,0.136760


SEMANTIC SEARCH


,recipe_id,title,source_link_raw,score
0,42e911d924b1e7909269b2d2ae3de6e5d762051a3731d8...,Creamy Tacos,www.cookbooks.com/Recipe-Details.aspx?id=972379,0.794732
1,bdaaa78527c2377a7dfe9412af77142ac05e674bc13a54...,Tacos,www.cookbooks.com/Recipe-Details.aspx?id=973834,0.794511
2,39b83015a79ddd72df63a3b76844ec15340522d1188d08...,Roll Your Own Soft Tacos,www.cookbooks.com/Recipe-Details.aspx?id=332135,0.777550
3,96dcc3ac15a033b2d5af9968087fdf116444f152f99891...,Creamy Tacos,www.cookbooks.com/Recipe-Details.aspx?id=768497,0.768976
4,4ba252a543d270c9bb191e4a16591c015fd8cd38703ca5...,Cream Tacos,www.cookbooks.com/Recipe-Details.aspx?id=651572,0.765064


In [11]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import hashlib
import sentence_transformers

INDEX_DIR = PROJECT_ROOT / "data" / "recipe_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

np.save(INDEX_DIR / "recipe_embeddings_v1.npy", recipe_embeddings)

search_df.to_json(
    INDEX_DIR / "recipe_documents_v1.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

text_hash = hashlib.sha256(
    json.dumps(
        search_df[["recipe_id", "search_text"]].to_dict("records"),
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

with (INDEX_DIR / "index_metadata_v1.json").open("w") as file:
    json.dump(
        {
            "embedding_model": EMBEDDING_MODEL_NAME,
            "normalized_embeddings": True,
            "recipe_count": len(search_df),
            "embedding_dimension": int(recipe_embeddings.shape[1]),
            "max_sequence_length": embedding_model.max_seq_length,
            "ordered_documents_sha256": text_hash,
            "sentence_transformers_version":
                sentence_transformers.__version__,
        },
        file,
        indent=2,
    )

print("Saved recipe documents, embeddings, and metadata.")

Saved recipe documents, embeddings, and metadata.


In [12]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
print("Current recipe CSV:", RECIPE_PATH)
print(f"CSV size: {RECIPE_PATH.stat().st_size:,} bytes")

print("\nCurrent search data:")
print("Search documents:", len(search_df))
print("Embedding shape:", recipe_embeddings.shape)

print("\nSaved candidate files:")
for filename in [
    "recipe_candidates_v1.jsonl",
    "recipe_candidates_filtered_v1.jsonl",
    "recipe_candidates_filtered_v2.jsonl",
]:
    path = PROJECT_ROOT / "data" / "processed" / filename

    if path.exists():
        with path.open(encoding="utf-8") as file:
            count = sum(1 for line in file if line.strip())
        print(f"{filename}: {count:,} recipes")
    else:
        print(f"{filename}: NOT FOUND")

print("\nCurrent search coverage:")
if "candidate_classes" in search_df.columns:
    for name in ["ramen", "sushi", "tacos", "spaghetti_bolognese"]:
        count = search_df["candidate_classes"].apply(
            lambda classes: name in classes
        ).sum()
        print(f"{name}: {count}")

Current recipe CSV: /home/users/al581/meal-memory/data/raw/full_dataset_complete.csv
CSV size: 2,294,981,083 bytes

Current search data:
Search documents: 3592
Embedding shape: (3592, 384)

Saved candidate files:
recipe_candidates_v1.jsonl: 3,998 recipes
recipe_candidates_filtered_v1.jsonl: 3,634 recipes
recipe_candidates_filtered_v2.jsonl: 3,592 recipes

Current search coverage:
ramen: 157
sushi: 177
tacos: 100
spaghetti_bolognese: 183


In [13]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
recipe_lookup = search_df.set_index("recipe_id")

for query in development_queries:
    print(f"\n{'=' * 75}\nQUERY: {query}")

    for method_name, search_function in [
        ("KEYWORD", keyword_search),
        ("SEMANTIC", semantic_search),
    ]:
        print(f"\n{method_name}")

        for rank, result in enumerate(
            search_function(query, top_k=3).to_dict("records"),
            start=1,
        ):
            recipe = recipe_lookup.loc[result["recipe_id"]]

            print(f"\n{rank}. {recipe['title']}")
            print("Ingredients:")
            for ingredient in recipe["ingredients"]:
                print(" -", ingredient)


QUERY: ramen with pork and mushrooms

KEYWORD

1. Ramen Pork ‘N’ Peppers
Ingredients:
 - 1 package (3 ounces) ramen noodles
 - 1 teaspoon cornstarch
 - 1/4 cup orange juice
 - 4-1/2 teaspoons reduced-sodium soy sauce
 - 1 tablespoon balsamic vinegar
 - 1-1/2 teaspoons honey
 - 1/8 teaspoon pepper
 - 1/2 cup each chopped green, sweet red and yellow pepper
 - 1-1/2 teaspoons canola oil
 - 1/2 pound cooked pork, cubed

2. Tokushima Ramen Pork Belly
Ingredients:
 - 300 to 400 grams Thinly sliced pork belly
 - 100 ml Soy sauce
 - 60 ml Mirin
 - 2 to 3 tablespoons Sugar (light brown sugar recommended)
 - 120 ml Sake

3. Bacon And Cabbage Ramen
Ingredients:
 - 1 (3 -4 ounce) package pork-flavored ramen noodles
 - 2 slices bacon
 - 2 ounces napa cabbage, sliced thin

SEMANTIC

1. Ramen Casserole
Ingredients:
 - 2 tablespoons sesame oil
 - 8 ounces fresh shiitake mushrooms, stems removed and caps thinly sliced
 - 2 (6-in.) square nori (dried seaweed) sheets, torn into small pieces
 - 1 (8-oz.)

In [14]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
def hybrid_search(query, top_k=5, rank_constant=60):
    """Combine full keyword and semantic rankings using rank fusion."""
    if not query.strip():
        raise ValueError("Enter a nonempty query.")

    # Keyword ranking.
    query_vector = tfidf_vectorizer.transform([query])
    keyword_scores = linear_kernel(
        query_vector, tfidf_matrix
    ).ravel()

    keyword_order = np.argsort(
        -keyword_scores, kind="stable"
    )

    # Only include keyword matches with positive similarity.
    keyword_order = keyword_order[
        keyword_scores[keyword_order] > 0
    ]

    # Semantic ranking.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    semantic_scores = recipe_embeddings @ query_embedding
    semantic_order = np.argsort(
        -semantic_scores, kind="stable"
    )

    # Reciprocal rank fusion: equal weight for both methods.
    fusion_scores = np.zeros(len(search_df), dtype=float)

    for ranking in [keyword_order, semantic_order]:
        ranks = np.arange(1, len(ranking) + 1)
        fusion_scores[ranking] += 1.0 / (rank_constant + ranks)

    order = np.argsort(
        -fusion_scores, kind="stable"
    )[:top_k]

    results = search_df.iloc[order][
        ["recipe_id", "title", "source_link_raw"]
    ].copy()
    results["score"] = fusion_scores[order]

    return results.reset_index(drop=True)


for query in development_queries:
    print(f"\nQUERY: {query}")
    display(hybrid_search(query))


QUERY: ramen with pork and mushrooms


,recipe_id,title,source_link_raw,score
0,7a10d61f92fad66fbd2d03b2ecad497d928c274d30a5eb...,Ramen Pork ‘N’ Peppers,www.tasteofhome.com/recipes/ramen-pork-n-peppers/,0.031319
1,39e80728e0916c291d623df8494a49a443d23479880b9c...,Black Pepper Pork With Ramen Noodles,www.food.com/recipe/black-pepper-pork-with-ram...,0.030090
2,9c1223039eb5cc2bd6795ecab1ee6936738ab3ebad1e5e...,Ramen Noodle Surprise,www.food.com/recipe/ramen-noodle-surprise-281064,0.028139
3,9a86281aeb2308240f14251bb5682715d97782a10eec96...,Asian Ramen Noodle Soup Recipe,cookeatshare.com/recipes/asian-ramen-noodle-so...,0.027783
4,5a40825f80355d3f5e6c249916ccc7e3c849db8dcc5ee3...,One Pot Fancy Chicken Ramen,cookpad.com/us/recipes/340280-one-pot-fancy-ch...,0.027501



QUERY: sushi with rice and vegetables


,recipe_id,title,source_link_raw,score
0,3183f1f58dab70ee1ceb6f466c4d83a122c7ea00983409...,Sushi Rice,www.epicurious.com/recipes/food/views/sushi-ri...,0.028992
1,e4dbd357fc33492f78c5e82027d228a51064f1e13b033c...,Make-Ahead Sushi Rice,www.food.com/recipe/make-ahead-sushi-rice-190284,0.027119
2,9b6022b45ef952d970323f56d2d21bc9c6b6271dc7ea92...,Sushi Rice,recipes-plus.com/api/v2.0/recipes/33222,0.026430
3,c0343c7f0101c71bea60b9be6b8d613e315c01ccd5a809...,Easy Sushi Rice for Hand-Rolled Sushi,cookpad.com/us/recipes/146093-easy-sushi-rice-...,0.026430
4,2f705f314b47d24882c878b5d11e11dc729b85bbb53afd...,Our Recipe For Sushi Rice,cookpad.com/us/recipes/188067-our-recipe-for-s...,0.026357



QUERY: tacos with beef and cheese


,recipe_id,title,source_link_raw,score
0,bdaaa78527c2377a7dfe9412af77142ac05e674bc13a54...,Tacos,www.cookbooks.com/Recipe-Details.aspx?id=973834,0.031281
1,4ba252a543d270c9bb191e4a16591c015fd8cd38703ca5...,Cream Tacos,www.cookbooks.com/Recipe-Details.aspx?id=651572,0.031258
2,39b83015a79ddd72df63a3b76844ec15340522d1188d08...,Roll Your Own Soft Tacos,www.cookbooks.com/Recipe-Details.aspx?id=332135,0.030366
3,1f61ff44b79a01494f95df3f05fb1675b97ac045438763...,Tico Tacos,www.cookbooks.com/Recipe-Details.aspx?id=204970,0.029710
4,e3d8abe28c68318a9d8a1cf20daaa8615267de7a1eb5b9...,Beef Tacos,www.cookbooks.com/Recipe-Details.aspx?id=200133,0.029211


In [15]:
# AI attribution: Evaluation queries drafted with assistance from OpenAI ChatGPT.
# These are constructed test scenarios, not collected user queries.
evaluation_query_pairs = {
    "apple_pie": [
        "a traditional apple pie with a pastry crust",
        "apple pie with cinnamon",
    ],
    "caesar_salad": [
        "a classic romaine Caesar salad",
        "Caesar salad topped with chicken",
    ],
    "cheesecake": [
        "a baked cheesecake with a crumb crust",
        "cheesecake with lemon",
    ],
    "chocolate_cake": [
        "a chocolate layer cake",
        "chocolate cake with chocolate frosting",
    ],
    "french_toast": [
        "French toast made in a frying pan",
        "French toast with berries",
    ],
    "fried_rice": [
        "fried rice containing egg",
        "fried rice with shrimp",
    ],
    "grilled_cheese_sandwich": [
        "a toasted sandwich with melted cheese",
        "a grilled cheese sandwich with tomato",
    ],
    "hamburger": [
        "a beef burger served on a bun",
        "a vegetarian bean burger",
    ],
    "lasagna": [
        "lasagna with meat and tomato sauce",
        "lasagna with spinach and cheese",
    ],
    "macaroni_and_cheese": [
        "baked macaroni and cheese",
        "macaroni and cheese with a breadcrumb topping",
    ],
    "omelette": [
        "an omelette with cheese",
        "an omelette with spinach",
    ],
    "pancakes": [
        "fluffy breakfast pancakes",
        "pancakes with blueberries",
    ],
    "pizza": [
        "pizza with tomato sauce and mozzarella",
        "pizza topped with mushrooms",
    ],
    "ramen": [
        "ramen noodle soup with an egg",
        "ramen with chicken and vegetables",
    ],
    "risotto": [
        "a creamy rice risotto",
        "risotto with asparagus",
    ],
    "spaghetti_bolognese": [
        "pasta with a Bolognese meat sauce",
        "Bolognese sauce made with ground turkey",
    ],
    "spaghetti_carbonara": [
        "spaghetti carbonara with egg and bacon",
        "carbonara pasta with mushrooms",
    ],
    "sushi": [
        "rice and nori sushi rolls",
        "sushi rolls with salmon",
    ],
    "tacos": [
        "chicken tacos served in tortillas",
        "tacos with beans and avocado",
    ],
    "waffles": [
        "breakfast waffles made in a waffle iron",
        "waffles with banana",
    ],
}

assert set(evaluation_query_pairs) == set(SELECTED_CLASSES)

evaluation_queries = [
    {
        "query_id": f"q{index:02d}",
        "class_name": class_name,
        "query": query,
    }
    for index, (class_name, query) in enumerate(
        (
            (name, query)
            for name in SELECTED_CLASSES
            for query in evaluation_query_pairs[name]
        ),
        start=1,
    )
]

EVAL_DIR = PROJECT_ROOT / "data" / "processed" / "retrieval_eval_v1"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

query_path = EVAL_DIR / "queries.json"
if query_path.exists():
    raise FileExistsError(
        "Evaluation files already exist. Stop before replacing them."
    )

with query_path.open("w") as file:
    json.dump(evaluation_queries, file, indent=2)

print("Saved evaluation queries:", len(evaluation_queries))

Saved evaluation queries: 40


In [16]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
methods = {
    "keyword": keyword_search,
    "semantic": semantic_search,
    "hybrid": hybrid_search,
}

rankings = []
pooled_pairs = {}

for query_record in evaluation_queries:
    query_id = query_record["query_id"]
    query = query_record["query"]

    for method_name, search_function in methods.items():
        results = search_function(query, top_k=5)

        for rank, result in enumerate(
            results.to_dict("records"), start=1
        ):
            recipe_id = result["recipe_id"]

            rankings.append({
                "query_id": query_id,
                "method": method_name,
                "rank": rank,
                "recipe_id": recipe_id,
            })

            recipe = recipe_lookup.loc[recipe_id]

            pooled_pairs[(query_id, recipe_id)] = {
                "query_id": query_id,
                "query": query,
                "recipe_id": recipe_id,
                "title": recipe["title"],
                "ingredients": "\n".join(recipe["ingredients"]),
                "directions": "\n".join(recipe["directions"]),
                "relevance": "",
                "notes": "",
            }

judging_sheet = (
    pd.DataFrame(pooled_pairs.values())
    .sample(frac=1, random_state=386)
    .reset_index(drop=True)
)

pd.DataFrame(rankings).to_csv(
    EVAL_DIR / "rankings.csv",
    index=False,
)
judging_sheet.to_csv(
    EVAL_DIR / "judgments_blank.csv",
    index=False,
)

print("Unique query–recipe pairs to judge:", len(judging_sheet))
print("Judging sheet:", EVAL_DIR / "judgments_blank.csv")

Unique query–recipe pairs to judge: 395
Judging sheet: /home/users/al581/meal-memory/data/processed/retrieval_eval_v1/judgments_blank.csv


In [17]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
from pathlib import Path
import pandas as pd

EVAL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "retrieval_eval_v1"
)
JUDGMENTS_PATH = EVAL_DIR / "judgments.csv"


def save_judgments(frame):
    """Replace the working CSV only after writing a complete new copy."""
    temporary_path = JUDGMENTS_PATH.with_suffix(".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(JUDGMENTS_PATH)


def grade_recipes(batch_size=10):
    """Grade pending recipes, saving each answer immediately."""
    template = pd.read_csv(
        EVAL_DIR / "judgments_blank.csv",
        dtype=str,
        keep_default_na=False,
    )

    if JUDGMENTS_PATH.exists():
        judgments = pd.read_csv(
            JUDGMENTS_PATH,
            dtype=str,
            keep_default_na=False,
        )
    else:
        judgments = template.copy()
        save_judgments(judgments)

    keys = ["query_id", "recipe_id"]

    assert not judgments.duplicated(keys).any()
    assert set(map(tuple, judgments[keys].to_numpy())) == set(
        map(tuple, template[keys].to_numpy())
    ), "Judging sheet differs from the original evaluation."

    assert judgments["relevance"].isin(
        ["", "0", "1", "2", "3"]
    ).all(), "Unexpected relevance values."

    pending = judgments.index[
        judgments["relevance"] == ""
    ].tolist()

    completed = len(judgments) - len(pending)
    print(f"Progress: {completed}/{len(judgments)} graded")

    if not pending:
        print("All judgments are complete.")
        return

    print(
        "\n0 = unrelated\n"
        "1 = related component or substantially different preparation\n"
        "2 = appropriate dish, but misses requested features\n"
        "3 = appropriate dish and requested features\n\n"
        "Enter a grade, optionally followed by | notes.\n"
        "Example: 2 | Has rice but no vegetables\n"
        "Enter q to stop. Previous answers are already saved."
    )

    for index in pending[:batch_size]:
        row = judgments.loc[index]

        print("\n" + "=" * 70)
        print("QUERY:", row["query"])
        print("\nRECIPE:", row["title"])
        print("\nINGREDIENTS:\n" + row["ingredients"])
        print("\nDIRECTIONS:\n" + row["directions"])

        while True:
            answer = input("\nGrade 0–3, or q: ").strip()

            if answer.lower() == "q":
                print("Stopped. All submitted grades are saved.")
                return

            grade, separator, notes = answer.partition("|")
            grade = grade.strip()

            if grade in {"0", "1", "2", "3"}:
                break

            print("Please enter 0, 1, 2, 3, or q.")

        judgments.at[index, "relevance"] = grade
        judgments.at[index, "notes"] = notes.strip()
        save_judgments(judgments)

        completed += 1
        print(f"Saved — {completed}/{len(judgments)} complete.")

    print("\nBatch complete. Run grade_recipes() for the next batch.")

In [18]:
grade_recipes(batch_size=10)

Progress: 0/395 graded

0 = unrelated
1 = related component or substantially different preparation
2 = appropriate dish, but misses requested features
3 = appropriate dish and requested features

Enter a grade, optionally followed by | notes.
Example: 2 | Has rice but no vegetables
Enter q to stop. Previous answers are already saved.

QUERY: a traditional apple pie with a pastry crust

RECIPE: Old Fashioned Apple Pie

INGREDIENTS:
4 c. apples, peeled and sliced
1 c. white sugar
1/2 c. brown sugar
3/4 tsp. cinnamon
1/2 tsp. nutmeg
3 Tbsp. tapioca
1/2 tsp. vanilla
2 tsp. butter
1 tsp. sugar
2 (9-inch) pie crusts

DIRECTIONS:
Preheat oven to 400°.
Combine together the first 7 ingredients.
Fill a 9-inch pie crust with the mixture.
Cut the butter into pieces and place on top of apple mixture.
Put 2nd pie crust on top.
Rub top crust with a little milk (this will make it crusty and brown).
Sprinkle a teaspoon of sugar on top crust. Bake for 45 to 50 minutes.



Grade 0–3, or q:  3


Saved — 1/395 complete.

QUERY: a toasted sandwich with melted cheese

RECIPE: " Almost Grilled" Cheese Sandwich

INGREDIENTS:
2 slices white bread or 2 slices whole wheat bread
1 slice processed cheese
butter

DIRECTIONS:
Toast the bread slices.
Spread one side of each bread slice very lightly with butter.
Put cheese slice between buttered sides of bread.
Heat in the microwave until cheese starts to melt.
In my oven this takes no more than 15 to 20 seconds.



Grade 0–3, or q:  3


Saved — 2/395 complete.

QUERY: Bolognese sauce made with ground turkey

RECIPE: Bolognese Sauce

INGREDIENTS:
14 cup olive oil
1 small onion, finely choped
2 celery ribs, finely chopped
1 carrot, finely chopped
1 lb ground chuck
salt
fresh ground pepper
1 cup whole milk
1 pinch nutmeg
12 cup dry white wine
1 (24 ounce) can Italian plum tomatoes, chopped

DIRECTIONS:
In a large pot, over medium heat, heat the oil.
Add the onion and cook for 10 minutes, or until the onion becomes soft.
Add the celery and carrot and continue to cook for 2 minutes.
Add the ground chuck and season to taste with the salt and pepper.
With the back of a spoon break up the meat and cook until the meat is just cooked through and no longer pink, about 5 minutes.
Carefully pour in the milk and reduce, stirring occasionally, until the milk has evaporated and the pan is dry.
Add the nutmeg and then add the wine.
Continue cooking until the wine has reduced by half, about 10-15 minutes.
To Finish the sauce, pour in t


Grade 0–3, or q:  2 | Bolognese sauce, but uses ground beef rather than ground turkey


Saved — 3/395 complete.

QUERY: pizza with tomato sauce and mozzarella

RECIPE: Pizza Cups

INGREDIENTS:
4 pkg. refrigerated biscuits
pizza or spaghetti sauce
Mozzarella cheese
browned sausage
ground beef
pepperoni
mushrooms
fennel seed
green pepper
onion

DIRECTIONS:
Brown meat.
Pat biscuits in bottom and sides of greased muffin pan.
Add other ingredients as desired.
Top with pizza or spaghetti sauce (approximately 2 tablespoons).
Then sprinkle with cheese and bake at 375° until biscuits are browned.



Grade 0–3, or q:  1 | Pizza-style biscuit cups rather than pizza; includes sauce and mozzarella


Saved — 4/395 complete.

QUERY: pancakes with blueberries

RECIPE: Blueberry Buttermilk Pancakes

INGREDIENTS:
1 1/3 cups all purpose flour
3 tablespoons sugar
2 1/2 teaspoons baking powder
3/4 teaspoon salt
1 1/4 cups buttermilk
2 large eggs
2 tablespoons (1/4 stick) unsalted butter, melted, plus more for cooking
2 cups fresh blueberries
Pure maple syrup

DIRECTIONS:
Preheat oven to 250°F. Place baking sheet in oven. Whisk first 4 ingredients in large bowl. Whisk buttermilk, eggs, and 2 tablespoons butter in medium bowl; stir into dry ingredients. Fold in berries.
Heat large nonstick griddle or skillet over medium heat; brush with butter. Drop batter by 1/3 cupfuls onto griddle. Cook pancakes until brown, brushing griddle with more butter as needed, about 3 minutes per side. Transfer to sheet in oven to keep warm. Serve pancakes with maple syrup.



Grade 0–3, or q:  q


Stopped. All submitted grades are saved.


In [19]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
import json
import random
from pathlib import Path

import pandas as pd

FULL_EVAL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "retrieval_eval_v1"
)
SMALL_EVAL_DIR = (
    PROJECT_ROOT / "data" / "processed" / "retrieval_audit_small_v1"
)

if SMALL_EVAL_DIR.exists():
    raise FileExistsError(
        "Small-audit folder already exists. Stop so we can reuse it."
    )

with (FULL_EVAL_DIR / "queries.json").open() as file:
    all_queries = json.load(file)

rng = random.Random(386)

categories = sorted({
    record["class_name"] for record in all_queries
})
selected_categories = sorted(rng.sample(categories, 10))

selected_queries = []
for category in selected_categories:
    options = [
        record for record in all_queries
        if record["class_name"] == category
    ]
    selected_queries.append(rng.choice(options))

selected_ids = {
    record["query_id"] for record in selected_queries
}

# Reuse the frozen rankings rather than rerunning search.
rankings = pd.read_csv(
    FULL_EVAL_DIR / "rankings.csv",
    dtype={"query_id": str, "recipe_id": str},
)

small_rankings = rankings[
    rankings["query_id"].isin(selected_ids)
    & (rankings["rank"] == 1)
].copy()

assert len(small_rankings) == 30, (
    "Expected one result per method for each of 10 queries."
)

pair_keys = small_rankings[
    ["query_id", "recipe_id"]
].drop_duplicates()

blank = pd.read_csv(
    FULL_EVAL_DIR / "judgments_blank.csv",
    dtype=str,
    keep_default_na=False,
)

small_blank = pair_keys.merge(
    blank,
    on=["query_id", "recipe_id"],
    how="left",
    validate="one_to_one",
)

assert small_blank["title"].notna().all()

small_blank = small_blank.sample(
    frac=1, random_state=386
).reset_index(drop=True)

working = small_blank.copy()
previous_path = FULL_EVAL_DIR / "judgments.csv"

if previous_path.exists():
    previous = pd.read_csv(
        previous_path,
        dtype=str,
        keep_default_na=False,
    ).set_index(["query_id", "recipe_id"])

    for index, row in working.iterrows():
        key = (row["query_id"], row["recipe_id"])
        if key in previous.index:
            working.at[index, "relevance"] = previous.loc[
                key, "relevance"
            ]
            working.at[index, "notes"] = previous.loc[key, "notes"]

SMALL_EVAL_DIR.mkdir(parents=True)

small_blank.to_csv(
    SMALL_EVAL_DIR / "judgments_blank.csv", index=False
)
working.to_csv(
    SMALL_EVAL_DIR / "judgments.csv", index=False
)
small_rankings.to_csv(
    SMALL_EVAL_DIR / "rankings.csv", index=False
)

with (SMALL_EVAL_DIR / "queries.json").open("w") as file:
    json.dump(selected_queries, file, indent=2)

# Point the existing grading interface at the SMALL audit.
EVAL_DIR = SMALL_EVAL_DIR
JUDGMENTS_PATH = SMALL_EVAL_DIR / "judgments.csv"

completed = working["relevance"].ne("").sum()
print("Queries:", len(selected_queries))
print("Unique pairs:", len(working))
print("Previous grades reused:", completed)
print("Remaining:", len(working) - completed)


Queries: 10
Unique pairs: 21
Previous grades reused: 0
Remaining: 21


In [20]:
grade_recipes(batch_size=10)

Progress: 0/21 graded

0 = unrelated
1 = related component or substantially different preparation
2 = appropriate dish, but misses requested features
3 = appropriate dish and requested features

Enter a grade, optionally followed by | notes.
Example: 2 | Has rice but no vegetables
Enter q to stop. Previous answers are already saved.

QUERY: a creamy rice risotto

RECIPE: Risotto With Lemon And Mint

INGREDIENTS:
6 cups water
salt to taste
3 tablespoons oil
1 chopped onion
2 cups rice for risotto
1 cup white wine
2 tablespoons lemon juice
1 tablespoon lemon zest
1 tablespoon mint chopped
2 tablespoons grated Parmesan cheese

DIRECTIONS:
Put water to boil with salt and let it boil.
Set aside covered.
In a pan put olive oil and the chopped onion and bring to heat until the onion is translucent.
Add the risotto and stir well until it gets golden.
Add the white wine and boil, stirring constantly, until it evaporates.
Mix the lemon juice, letting it absorb.
Add slowly the water with salt, st


Grade 0–3, or q:  3


Saved — 1/21 complete.

QUERY: pizza with tomato sauce and mozzarella

RECIPE: Pizza Cups

INGREDIENTS:
1 lb. ground meat
1 small can tomato sauce
1/4 tsp. Italian seasoning (optional)
1/2 tsp. salt and pepper
1 can biscuits
1/2 to 3/4 c. grated Mozzarella cheese

DIRECTIONS:
Cool meat and put biscuits into muffin tins with mix and cheese on top.
Bake at 400° for 12 minutes or until golden brown.



Grade 0–3, or q:  1 | Pizza-style biscuit cups rather than pizza; includes tomato sauce and mozzarella


Saved — 2/21 complete.

QUERY: pizza with tomato sauce and mozzarella

RECIPE: Pizza Chicken

INGREDIENTS:
6 boneless and skinless chicken breasts
1 (15 oz.) jar pizza sauce
1 c. Mozzarella cheese

DIRECTIONS:
Place chicken in baking dish and bake at 350° for 30 minutes. Pour pizza sauce over top and bake 30 minutes longer.
Sprinkle with cheese.
Bake for additional 5 minutes or until cheese is melted.
Serves 6.



Grade 0–3, or q:  1 | Pizza-style chicken rather than pizza; includes pizza sauce and mozzarella


... (11 outputs left out)


Grade 0–3, or q:  3


Saved — 9/21 complete.

QUERY: French toast with berries

RECIPE: French Toast

INGREDIENTS:
2 eggs
1/2 c. milk
1 Tbsp. cinnamon
1 tsp. vanilla

DIRECTIONS:
Mix together.
Dip sliced bread in mixture and fry on each side until golden brown on both sides.
Sprinkle a lot of powdered sugar on top of each slice.
May be also served with syrup.



Grade 0–3, or q:  2 | French toast, but no berries are included


Saved — 10/21 complete.

Batch complete. Run grade_recipes() for the next batch.


In [21]:
grade_recipes(batch_size=11)

Progress: 10/21 graded

0 = unrelated
1 = related component or substantially different preparation
2 = appropriate dish, but misses requested features
3 = appropriate dish and requested features

Enter a grade, optionally followed by | notes.
Example: 2 | Has rice but no vegetables
Enter q to stop. Previous answers are already saved.

QUERY: ramen with chicken and vegetables

RECIPE: Boiled Chicken Ramen Recipe

INGREDIENTS:
1 pkt of ramen - chicken flavored
1 pkt of ramen - beef flavored some pre-boiled chicken

DIRECTIONS:
Make your ramen.
Stick in your chicken and let it all cook till the noodles are done and the chicken is re-heated.
Then toss in yer flavor packets.
MMMM.



Grade 0–3, or q:  2 | Ramen with chicken, but no vegetables are included


Saved — 11/21 complete.

QUERY: spaghetti carbonara with egg and bacon

RECIPE: Spaghetti Carbonara

INGREDIENTS:
8 -12 thick slices bacon (Italian bacon) or 8 -12 thick slices pancetta (Italian bacon)
2 garlic cloves, minced
1 onion, chopped
1 teaspoon crushed red pepper flakes (optional)
4 eggs, beaten
salt & fresh ground pepper
2 (16 ounce) packages spaghetti
2 cups parmesan cheese, freshly grated

DIRECTIONS:
Cut up bacon/pancetta into small squares; add to a saute pan along with garlic and onion cook over medium high heat.
Simmer for 30 minutes; do not let mixture boil.
During last 10 minutes of simmering add crushed red pepper flakes if using.
In a bowl, beat 4 eggs with salt and fresh ground pepper.
Meanwhile cook spaghetti according to package.
Drain and immediately add egg mixture to pasta.
(The pasta needs to be really hot in order for the eggs to cook).
Stir until eggs are cooked.
Mix into saute pan and toss with the grated Parmesan cheese until combined.



Grade 0–3, or q:  3


Saved — 12/21 complete.

QUERY: ramen with chicken and vegetables

RECIPE: One Pot Fancy Chicken Ramen

INGREDIENTS:
2 Strips of bacon cut into small pieces
1 can Chicken breast (12.5 oz) drained
1 tbsp Diced onion
1 Bell pepper
1 can Mushrooms (4 oz) NOT drained
1 can Diced tomatoes (14.5 oz) NOT drained
1/2 cup Water
1 envelope Chicken Ramen Noodle flavoring
1 dash Pepper
1 packages Ramen noodles
1/3 box Cream cheese cubed
1 can Cream of chicken soup

DIRECTIONS:
Cook bacon in a large sauce pan
Stir in onion and bell pepper.
I use this much bell pepper
Add chicken and stir till it looks shredded.
Add undrained tomatoes and mushrooms.
Add water and enough milk to have the liquid needed to cook the noodles.
Stir together.
Put packet of chicken flavor in from package of noodles.
Sprinkle pepper over mixture.
Cook until it starts to boil.
Split your noodles as shown below then break each square into quarters.
Stir into boiling sauce.
Keep stirring until noodles are done.
Then add cream c


Grade 0–3, or q:  3


... (13 outputs left out)


Grade 0–3, or q:  3


Saved — 20/21 complete.

QUERY: apple pie with cinnamon

RECIPE: Apple Pie

INGREDIENTS:
5 to 6 apples, peeled and sliced
1 c. sugar
1/4 c. flour
2/3 c. margarine
1 tsp. cinnamon

DIRECTIONS:
Arrange apples in pie crust; sprinkle 1/2 cup sugar over apples.
Combine remaining 1/2 cup sugar with flour and cinnamon. Sprinkle mixture over apples.
Add margarine to top of mixture and place top crust over all.
Bake in microwave on High for 7 minutes.
Then bake in 350° oven until brown.



Grade 0–3, or q:  3


Saved — 21/21 complete.

Batch complete. Run grade_recipes() for the next batch.


In [22]:
# AI attribution: Drafted with assistance from OpenAI ChatGPT.
AUDIT_DIR = (
    PROJECT_ROOT / "data" / "processed"
    / "retrieval_audit_small_v1"
)

judgments = pd.read_csv(
    AUDIT_DIR / "judgments.csv",
    dtype=str,
    keep_default_na=False,
)
rankings = pd.read_csv(
    AUDIT_DIR / "rankings.csv",
    dtype={"query_id": str, "recipe_id": str},
)

keys = ["query_id", "recipe_id"]

assert not judgments.duplicated(keys).any()
assert judgments["relevance"].isin(["0", "1", "2", "3"]).all(), (
    "Some judgments are missing or invalid."
)
assert not rankings.duplicated(["query_id", "method"]).any()
assert rankings["rank"].eq(1).all()

judgments["relevance"] = judgments["relevance"].astype(int)

scored_results = rankings.merge(
    judgments[keys + ["relevance"]],
    on=keys,
    how="left",
    validate="many_to_one",
)

assert scored_results["relevance"].notna().all()

expected_methods = {"keyword", "semantic", "hybrid"}
assert set(scored_results["method"]) == expected_methods

query_sets = [
    set(group["query_id"])
    for _, group in scored_results.groupby("method")
]
assert all(query_set == query_sets[0] for query_set in query_sets)
assert len(query_sets[0]) == 10

scored_results["full_match"] = (
    scored_results["relevance"] == 3
)
scored_results["acceptable_match"] = (
    scored_results["relevance"] >= 2
)

audit_summary = (
    scored_results.groupby("method")
    .agg(
        queries=("query_id", "nunique"),
        full_match_rate=("full_match", "mean"),
        acceptable_match_rate=("acceptable_match", "mean"),
        mean_relevance=("relevance", "mean"),
    )
    .reset_index()
)

print("Rates are proportions: 0.80 means 80%.")
display(audit_summary)

# Save compact evaluation evidence without copying recipe text.
EVIDENCE_DIR = PROJECT_ROOT / "docs" / "results" / "retrieval_audit"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

audit_summary.to_csv(
    EVIDENCE_DIR / "summary.csv", index=False
)
scored_results.to_csv(
    EVIDENCE_DIR / "per_query_results.csv", index=False
)
judgments[keys + ["relevance", "notes"]].to_csv(
    EVIDENCE_DIR / "judgments.csv", index=False
)

with (AUDIT_DIR / "queries.json").open() as file:
    audit_queries = json.load(file)

with (EVIDENCE_DIR / "queries.json").open("w") as file:
    json.dump(audit_queries, file, indent=2)

print("Saved evaluation evidence to:", EVIDENCE_DIR)

Rates are proportions: 0.80 means 80%.


,method,queries,full_match_rate,acceptable_match_rate,mean_relevance
0,hybrid,10,0.9,0.9,2.8
1,keyword,10,0.6,0.8,2.4
2,semantic,10,0.9,0.9,2.8


Saved evaluation evidence to: /home/users/al581/meal-memory/docs/results/retrieval_audit


## Retrieval Evaluation and Design Decision

I compared TF-IDF keyword search, MiniLM semantic search, and
reciprocal-rank-fusion hybrid search over the same 3,592 recipes.

In an exploratory top-one audit of 10 constructed queries, semantic
and hybrid search each achieved 9/10 full matches and mean relevance
of 2.8/3. Keyword search achieved 6/10 full matches, 8/10 acceptable
matches, and mean relevance of 2.4/3.

I selected semantic search as the application default because hybrid
search provided no measured advantage while requiring an additional
retrieval component.

Limitations: this was a small, single-reviewer, AI-assisted audit.
Results are exploratory and do not establish general superiority.

The earlier ramen, sushi, and taco examples were development queries,
separate from this audit.